<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Cascades

**[Cascades](https://learning.nextia-ai.com/courses/text-routing/m06/cascades/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 6, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build a cascade: a cheap model decides the messages it is sure about, an LLM decides the
next ones, and a person reads the rest. You choose the two thresholds on the validation messages and score the
choice once on the test messages.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the LLM's answers were recorded once. |
| **You should know** | TF-IDF + logistic regression, from [Naive Bayes, logistic regression and linear SVM](https://learning.nextia-ai.com/courses/text-routing/m03/naive-bayes-logistic-regression-and-linear-svm/); the LLM with examples, from [Few-shot examples and label descriptions](https://learning.nextia-ai.com/courses/text-routing/m05/few-shot-examples-and-label-descriptions/). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m06/cascades/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M06-L03-cascades/cascades-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "ec1d7769d1d68a9e51a5b12660364c74352af936df79533a0d2170867171bdd1",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "04df0cd72da4e6b7e6aa629ca21205e5395df7b66aae9268618ae14597e45782",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "54c2e724cb3c855a0fa6302d4ac0ab2df2ffaf788eec4ab198c6feb8d3508928",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The two models

The **first step** is cheap: TF-IDF + logistic regression, the project's model, trained on the 6,000 training
messages. It gives a probability for each route. The **second step** is the hosted LLM with examples. Its answers
for the validation and test messages were recorded once, with the confidence that the LLM wrote. Replaying them
costs nothing, but a real call costs about US$0.088 per 1,000 messages.

The next cell trains the first step and loads the recorded answers of the second.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline

features = FeatureUnion([
    ("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")),
    ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True))])
model = make_pipeline(features, LogisticRegression(C=10, max_iter=3000)).fit(train["text"], train["route"])

def first_step(part):
    """The cheap model's route and its top probability for each message."""
    proba = model.predict_proba(part["text"])
    return model.classes_[proba.argmax(1)], proba.max(1)

llm_val, meta = rt.predictions("llm-few-shot", validation)
llm_test, _ = rt.predictions("llm-few-shot", test)
print(meta["method"], "-", meta["run_per_1000"], "dollars per 1,000 messages")

> **Check.** You should see `hosted LLM chat-small, few-shot (recorded) - 0.0876 dollars per 1,000 messages`.

## 2. One cascade, three messages

A cascade asks two questions for each message:

1. Is the cheap model **sure**: is its top probability at least `sure`? Then it decides.
2. If not, is the LLM sure: did it write a confidence of at least `llm_sure`? Then the LLM decides.
3. If not, a person reads the message (`review`, US$1.50).

The function below does this for many messages at once. The next cell runs it on three test messages, with
`sure = 0.80` and `llm_sure = 0.95`.

In [ ]:
def cascade(tf_route, tf_p, llm_route, llm_conf, sure, llm_sure):
    """The route (or "review") and the step that decided, for each message."""
    step = np.where(tf_p >= sure, "tfidf", np.where(llm_conf >= llm_sure, "llm", "person"))
    route = np.where(step == "tfidf", tf_route, np.where(step == "llm", llm_route, "review"))
    return route, step

three = test[test["message_id"].isin(["M-100001", "M-100561", "M-100005"])]
r3, p3 = first_step(three)
l3 = llm_test.set_index("message_id").loc[three["message_id"]]
route, step = cascade(r3, p3, l3["prediction"].to_numpy(), l3["confidence"].to_numpy(), 0.80, 0.95)
pd.DataFrame({"message_id": three["message_id"], "label": three["route"], "tfidf": r3, "tfidf_p": p3.round(3),
              "llm": l3["prediction"].to_numpy(), "llm_conf": l3["confidence"].to_numpy(), "step": step, "route": route})

> **Check.** You should see three rows. `M-100001`: TF-IDF is sure (1.0) and decides `delivery`. `M-100561`: TF-IDF
> is not sure (0.593), and the LLM decides `orders` at 0.99. `M-100005`: both are unsure (0.416 and 0.62), so the
> message goes to `review`.

## 3. The cost of a cascade

The cost per 1,000 messages has two parts: the mistakes and reviews (the course's cost table), and the running
cost. Only the messages that reach the LLM cost money to run.

> **Predict.** With `sure = 0.80` and `llm_sure = 0.95`, how many of the 2,000 validation messages does each step
> decide?

In [ ]:
def cascade_cost(part, route, step):
    """Dollars per 1,000 messages: mistakes and reviews, plus the LLM calls."""
    mistakes = rt.cost_per_1000(part, route)
    running = (step != "tfidf").mean() * meta["run_per_1000"]
    return round(mistakes + running, 1)

vr, vp = first_step(validation)
lv_r, lv_c = llm_val["prediction"].to_numpy(), llm_val["confidence"].to_numpy()
route, step = cascade(vr, vp, lv_r, lv_c, 0.80, 0.95)
print(pd.Series(step).value_counts().to_dict())
print("validation:", cascade_cost(validation, route, step), "dollars per 1,000")

> **Check.** You should see `{'tfidf': 1702, 'llm': 213, 'person': 85}` and `103.8`.

## 4. Choose the two thresholds on validation

Now try every pair of thresholds on a grid: `sure` from 0.30 to 1.00 in steps of 0.05, and `llm_sure` either 0
(the LLM decides everything it gets) or a value on the same grid. Keep the cheapest pair. Use the validation
messages only: the test messages must stay unseen until the choice is made.

In [ ]:
grid = np.round(np.arange(0.30, 1.0001, 0.05), 2)
rows = []
for sure in grid:
    for llm_sure in np.concatenate([[0.0], grid]):
        r, s = cascade(vr, vp, lv_r, lv_c, sure, llm_sure)
        rows.append({"sure": sure, "llm_sure": llm_sure, "total": cascade_cost(validation, r, s)})
search = pd.DataFrame(rows)
best = search.loc[search["total"].idxmin()]
print(best.to_dict())
search.pivot(index="sure", columns="llm_sure", values="total").loc[[0.5, 0.7, 0.8, 0.9], [0.0, 0.8, 0.9, 0.95]]

> **Check.** You should see `{'sure': 0.8, 'llm_sure': 0.95, 'total': 103.8}`. In the small table, the
> column `0.0` is "the LLM decides everything it gets": every value in it is higher than in the column `0.95`.

## 5. Score the choice once on test

In [ ]:
tr_, tp = first_step(test)
lt_r, lt_c = llm_test["prediction"].to_numpy(), llm_test["confidence"].to_numpy()
route, step = cascade(tr_, tp, lt_r, lt_c, 0.80, 0.95)
decided = route != "review"
print(pd.Series(step).value_counts(normalize=True).round(3).to_dict())
print("accuracy of the decided:", round((route[decided] == test["route"].to_numpy()[decided]).mean(), 3))
print("test:", cascade_cost(test, route, step), "dollars per 1,000")

> **Check.** You should see TF-IDF 85.9%, the LLM 10.0%, a person 4.2% (as shares 0.859, 0.1
> and 0.042), an accuracy of 0.991 and **94.3** dollars per 1,000. The LLM for every message costs 116.1,
> and TF-IDF alone 296.0.

## 6. Your turn: the same cascade without the LLM

> **Your turn.** Keep the first step at `sure = 0.80`, but send every message that TF-IDF is not sure about straight
> to a person: no LLM. (Hint: a written confidence is never above 1, so `llm_sure = 2.0` sends every LLM answer to
> a person.) Put its cost on the **validation** messages in `no_llm_cost`. Before you run it, predict: is the LLM
> step worth its calls?

In [ ]:
no_llm_cost = ...  # the cost per 1,000 validation messages of TF-IDF at 0.80, and a person for the rest

In [ ]:
expect('no_llm_cost', no_llm_cost, 235.5, tolerance=0.001)

Without the LLM, 85 + 213 = 298 messages go to a person, and the cost on validation is
235.5, against 103.8 with the LLM. On test, it is 227.5 against 94.3. The LLM is
worth its calls here: it is right on most of the messages that TF-IDF is not sure about, and a review costs much
more than a call.

## 7. Failure case: choosing on the test messages

Tomás tries every `sure` from 0.30 to 1.00 in steps of 0.01 **on the test messages**, with `llm_sure = 0.95`, and
finds a lower cost than ours.

In [ ]:
fine = np.round(np.arange(0.30, 1.0001, 0.01), 2)
on_test = {s: cascade_cost(test, *cascade(tr_, tp, lt_r, lt_c, s, 0.95)) for s in fine}
t_best = min(on_test, key=on_test.get)
on_val = cascade_cost(validation, *cascade(vr, vp, lv_r, lv_c, t_best, 0.95))
print("best on test:", t_best, on_test[t_best], "- the same threshold on validation:", on_val)

> **Check.** You should see `best on test: 0.73 89.0 - the same threshold on validation:
> 225.8`.

The threshold 0.73 looks better on test (89.0 against 94.3), because it was chosen to fit
the test messages. On the validation messages, the same threshold costs 225.8. A few expensive
messages decide where the lowest point falls. A threshold chosen on the test set gives a number that the next
month will not show. Choose on validation, and report the test number once, even when it is not the lowest.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m06/cascades/) has the explorer, the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Distillation: let the LLM teach a cheap model](https://learning.nextia-ai.com/courses/text-routing/m06/distillation-let-the-llm-teach-a-cheap-model/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m06/cascades/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.